In [0]:
%run /Configurations/Init_Scripts

In [0]:
#Reference Table from Snowflake for Invoices = AES_DW.OUTBOUND.DATABRICKS_INVOICE_DETAILS_VW
#Role for DEV : ROLE_AES_DW_DEV_RO  and Role for PROD : ROLE_AES_DW_RO
 
database_host_url = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="SnowflakeURL")
username = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="SnowflakeUserName")
password = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="SnowflakePassword")
# database_name = "AES_DW"
schema_name = "OUTBOUND"
warehouse_name = "AES_ELT_WH_DEV"
table_name = "DATABRICKS_INVOICE_DETAILS_VW"
# deltaTablePath = 'dbfs:/mnt/silver/DIMConsumer'

CreatedBy = dbutils.widgets.text("CreatedBy","ADB_SFLoad")
CreatedBy = dbutils.widgets.get('CreatedBy')

UpdatedBy = dbutils.widgets.text("UpdatedBy","ADB_SFLoad")
UpdatedBy = dbutils.widgets.get('UpdatedBy')

dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")
 
database_name = dbutils.widgets.text("database_name","AES_DW")
database_name = dbutils.widgets.get('database_name')

destination_table = dbutils.widgets.text("destination_table","promotion.fact_invoiceaddendum")
destination_table = dbutils.widgets.get('destination_table')

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')

dbutils.widgets.text('ExternalLocation_silver',"/mntprod_silver")
ExternalLocation_silver = dbutils.widgets.get('ExternalLocation_silver')

dbutils.widgets.text('destination_path',"/FACTInvoiceAddendum")
destination_path = dbutils.widgets.get('destination_path')
destination_path=ExternalLocation_silver+destination_path

dbutils.widgets.text('role_name',"ROLE_AES_DW_RO")
role_name = dbutils.widgets.get('role_name')

dbutils.widgets.text('max_date',"2024-06-27")
max_date = dbutils.widgets.get('max_date')


df_source = (spark.read
      .format("snowflake")
      .option("dbtable", table_name)
      .option("sfUrl", database_host_url)
      .option("sfUser", username)
      .option("sfPassword", password)
      .option("sfDatabase", database_name)
      .option("sfSchema", schema_name)
      .option("sfWarehouse", warehouse_name)
      .option("sfRole", role_name)
      .load()
    )#.select("order_number","invoice_number","invoice_date")

df_source.createOrReplaceTempView("source_table")

In [0]:
df_source = spark.sql(""" 
SELECT 
    order_number AS Sales_Order, 
    invoice_number,
    row_number() OVER (PARTITION BY order_number  ORDER BY total_amount DESC) rn
FROM (
    SELECT  
        order_number, 
        invoice_number,
        SUM(amount) AS total_amount
    FROM 
        source_table
    WHERE 
        invoice_date >= '2024-04-28' 
        AND order_number NOT LIKE '030%'
        AND product_code IN ('20084559','20084560','20084561','20084562')
    GROUP BY 
        order_number, 
        invoice_number
        )
 
QUALIFY     row_number() OVER (PARTITION BY order_number ORDER BY total_amount DESC) = 1
""")

# df_source = df_source.withColumn("Bill_Date", to_date("Bill_Date"))
df_source = df_source.dropDuplicates()

In [0]:
Deltatbl_UL_Logs = DeltaTable.forPath(spark, destination_path)

(Deltatbl_UL_Logs.alias("tgt")
    .merge(df_source.alias("src"),
           "tgt.SalesOrderNumber = src.Sales_Order  and tgt.InvoiceNumber is  null")
    .whenMatchedUpdate(set={
        "InvoiceNumber": col("src.invoice_number"),
        "UpdatedBy": lit("ADB_SFInvoiceNumber"),
        "UpdatedDate": lit(current_timestamp())
    })
    .execute()
)

In [0]:
# # Read Data from the source
# df_source = (
#     spark.read
#     .format("snowflake")
#     .option("dbtable", table_name)
#     .option("sfUrl", database_host_url)
#     .option("sfUser", username)
#     .option("sfPassword", password)
#     .option("sfDatabase", database_name)
#     .option("sfSchema", schema_name)
#     .option("sfWarehouse", warehouse_name)
#     .option("sfRole", role_name)
#     .load()
# )
# df_source.createOrReplaceTempView("source_table")

# df_source = spark.sql("""
#     SELECT 
#         customer_shipto_id, 
#         order_number AS Sales_Order, 
#         invoice_number, 
#         Bill_Date
#     FROM (
#         SELECT  
#             customer_shipto_id, 
#             customer_soldto_id, 
#             order_number, 
#             invoice_number, 
#             invoice_date,  
#             CASE 
#                 WHEN DAY(invoice_date) >= 28 THEN DATEADD(DAY, 28 - DAY(invoice_date), invoice_date)
#                 ELSE DATEADD(MONTH, -1, DATEADD(DAY, 28 - DAY(invoice_date), invoice_date))
#             END AS Bill_Date,
#             SUM(amount) AS total_amount
#         FROM source_table
#         WHERE 
#             invoice_date >= '2024-04-28' AND 
#             order_number NOT LIKE '030%' AND 
#             product_code IN ('20084559', '20084560', '20084561', '20084562')
#         GROUP BY 
#             customer_shipto_id, 
#             customer_soldto_id, 
#             order_number, 
#             invoice_number, 
#             invoice_date
#     )
#     QUALIFY 
#         ROW_NUMBER() OVER (PARTITION BY customer_shipto_id, Bill_Date ORDER BY total_amount DESC) = 1
# """)

# df_source = df_source.drop_duplicates()
# df_source = df_source.withColumn("Bill_Date", to_date("Bill_Date"))
# df_source.createOrReplaceTempView("df_source")

# df_source = spark.sql("""
#     SELECT 
#         df_source.*, 
#         temp.ShipToAccountUUID 
#     FROM df_source  
#     LEFT JOIN (
#         SELECT DISTINCT  
#             a.InvoiceDate, 
#             b.ShipToAccountId, 
#             a.ShipToAccountUUID 
#         FROM promotion.fact_invoiceaddendum a 
#         LEFT JOIN promotion.dim_account b  
#         ON a.ShipToAccountUUID = b.ShipToAccountUUID
#     ) temp
#     ON df_source.customer_shipto_id = temp.ShipToAccountId  
#     AND temp.InvoiceDate = df_source.Bill_Date
# """)

In [0]:
# # Perform merge operation
# Deltatbl_UL_Logs = DeltaTable.forPath(spark, destination_path)

# (Deltatbl_UL_Logs.alias("tgt")
#     .merge(df_source.alias("src"),
#            f"tgt.ShipToAccountUUID = src.ShipToAccountUUID and tgt.InvoiceDate = src.Bill_Date and tgt.InvoiceDate >= '{max_date}' and tgt.SalesOrderNumber is null  and tgt.InvoiceNumber is  null ")
#     .whenMatchedUpdate(set={
#         "SalesOrderNumber": "src.Sales_Order",
#         "InvoiceNumber": "src.invoice_number",
#         "UpdatedBy": lit("ADB_SFSalesInvoiceNumber"),
#         "UpdatedDate": lit(current_timestamp())
#     })
#     .execute()
# )